# StrataForge Progress Notebook

## Phase 02 Tree Pipeline

Purpose: execute the deterministic Phase 02 hierarchy builder against committed parse-artifact fixtures and inspect committed node cards, verification status, and explicit unassigned spans.


### Environment Assumptions

- Run `uv sync --extra dev` from the repository root.
- The notebook consumes committed fixtures under `fixtures/phase02/inputs/` only.
- Fixture copies and executed tree outputs are written under `notebooks/_artifacts/phase02-notebook/`.
- No OCR or PDF re-parsing is required for the Phase 02 notebook path.


In [1]:
# environment setup
import json
import platform
import shutil
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    candidate = REPO_ROOT.parent
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print({
    "repo_root": str(REPO_ROOT),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "phase02_fixture_root": str(REPO_ROOT / "fixtures" / "phase02" / "inputs"),
})


{'repo_root': '/home/pruthvi/projects/StrataForge', 'python': '3.12.12', 'platform': 'Linux-6.12.62+rpt-rpi-2712-aarch64-with-glibc2.41', 'phase02_fixture_root': '/home/pruthvi/projects/StrataForge/fixtures/phase02/inputs'}


In [2]:
# imports
from strataforge.domain import TreeBuildRequest
from strataforge.tree import NoopRepairEngine, build_tree


In [3]:
# configuration
NOTEBOOK_ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "phase02-notebook"
FIXTURE_SOURCE_ROOT = REPO_ROOT / "fixtures" / "phase02" / "inputs"
FIXTURE_COPY_ROOT = NOTEBOOK_ARTIFACT_ROOT / "fixtures"
shutil.rmtree(NOTEBOOK_ARTIFACT_ROOT, ignore_errors=True)
FIXTURE_COPY_ROOT.mkdir(parents=True, exist_ok=True)

fixture_names = ("clean_outline", "unassigned_pages")
fixture_manifests = {}
for fixture_name in fixture_names:
    source_root = FIXTURE_SOURCE_ROOT / fixture_name
    copied_root = FIXTURE_COPY_ROOT / fixture_name
    shutil.copytree(source_root, copied_root)
    fixture_manifests[fixture_name] = copied_root / "manifest.json"

fixture_manifests


{'clean_outline': PosixPath('/home/pruthvi/projects/StrataForge/notebooks/_artifacts/phase02-notebook/fixtures/clean_outline/manifest.json'),
 'unassigned_pages': PosixPath('/home/pruthvi/projects/StrataForge/notebooks/_artifacts/phase02-notebook/fixtures/unassigned_pages/manifest.json')}

In [4]:
# execution
results = {}
for fixture_name, parse_manifest_path in fixture_manifests.items():
    tree_manifest = build_tree(
        TreeBuildRequest(
            parse_manifest_path=str(parse_manifest_path),
            tree_run_id=f"notebook-{fixture_name}",
        ),
        repair_engine=NoopRepairEngine(),
    )
    committed_cards = json.loads(Path(tree_manifest.node_cards_path).read_text(encoding="utf-8"))
    verification = json.loads(Path(tree_manifest.verification_report_path).read_text(encoding="utf-8"))
    unassigned = json.loads(Path(tree_manifest.unassigned_spans_path).read_text(encoding="utf-8"))
    results[fixture_name] = {
        "committed_titles": [card["title"] for card in committed_cards],
        "verification_status": verification["status"],
        "unassigned_spans": unassigned,
        "artifact_root": tree_manifest.artifact_root,
    }

results


{'clean_outline': {'committed_titles': ['Overview', 'Details', 'Appendix'],
  'verification_status': 'passed',
  'unassigned_spans': [],
  'artifact_root': '/home/pruthvi/projects/StrataForge/notebooks/_artifacts/phase02-notebook/fixtures/clean_outline/tree/notebook-clean_outline'},
 'unassigned_pages': {'committed_titles': ['1 Start', '2 End'],
  'verification_status': 'passed',
  'unassigned_spans': [{'document_id': '6666666666666666666666666666666666666666666666666666666666666666',
    'page_span': {'end_page': 0, 'start_page': 0},
    'reason': 'before_first_heading'},
   {'document_id': '6666666666666666666666666666666666666666666666666666666666666666',
    'page_span': {'end_page': 2, 'start_page': 2},
    'reason': 'between_verified_nodes'},
   {'document_id': '6666666666666666666666666666666666666666666666666666666666666666',
    'page_span': {'end_page': 4, 'start_page': 4},
    'reason': 'after_last_heading'}],
  'artifact_root': '/home/pruthvi/projects/StrataForge/notebooks/

In [5]:
# inspect results
for fixture_name, result in results.items():
    print(f"Fixture: {fixture_name}")
    print(f"  committed titles: {result['committed_titles']}")
    print(f"  verification: {result['verification_status']}")
    print(f"  unassigned spans: {result['unassigned_spans']}")
    print()


Fixture: clean_outline
  committed titles: ['Overview', 'Details', 'Appendix']
  verification: passed
  unassigned spans: []

Fixture: unassigned_pages
  committed titles: ['1 Start', '2 End']
  verification: passed
  unassigned spans: [{'document_id': '6666666666666666666666666666666666666666666666666666666666666666', 'page_span': {'end_page': 0, 'start_page': 0}, 'reason': 'before_first_heading'}, {'document_id': '6666666666666666666666666666666666666666666666666666666666666666', 'page_span': {'end_page': 2, 'start_page': 2}, 'reason': 'between_verified_nodes'}, {'document_id': '6666666666666666666666666666666666666666666666666666666666666666', 'page_span': {'end_page': 4, 'start_page': 4}, 'reason': 'after_last_heading'}]



### Known Limitations

- Phase 02 repair remains deterministic and no-op by default.
- The notebook intentionally exercises committed parse-artifact fixtures, not live PDF parsing.
- Tree-run indexing is now namespace-global within the artifact root, but registry writes are still not concurrency-safe.
